AE + BGWO for feature selection + SVM classifier on the selected features

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Data Loading & Preparation
df = pd.read_excel('../minmax.xlsx') 
data = df.values
labels = pd.read_csv('../idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model
class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh() 
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer
reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance losses
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

# Initialize model, optimizer, and scheduler
model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training (Pretraining Stage)
print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features Using the Trained Encoder
model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train).numpy()
    test_latent = model.encoder(X_test).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

# 6. BGWO Implementation for Feature Selection
def bgwo_optimize(features, labels, num_wolves=10, max_iterations=20):
    num_features = features.shape[1]
    wolves = np.random.randint(0, 2, size=(num_wolves, num_features))  # Binary positions
    # Ensure each wolf selects at least one feature
    for wolf in wolves:
        if not wolf.any():
            wolf[np.random.randint(0, num_features)] = 1

    alpha, beta, delta = None, None, None

    def fitness_function(position):
        selected_features = [i for i in range(len(position)) if position[i] > 0.5]
        if len(selected_features) == 0:  # Penalize empty feature subsets
            return -1  # Return a very low fitness score
        model = SVC(random_state=42)
        model.fit(features[:, selected_features], labels)
        preds = model.predict(features[:, selected_features])
        return accuracy_score(labels, preds)

    for iteration in range(max_iterations):
        fitness = np.array([fitness_function(wolf) for wolf in wolves])
        sorted_indices = np.argsort(fitness)[::-1]
        alpha, beta, delta = wolves[sorted_indices[:3]]

        for i in range(num_wolves):
            for j in range(num_features):
                r1, r2, r3 = np.random.rand(3)
                A1, A2, A3 = 2 * r1 - 1, 2 * r2 - 1, 2 * r3 - 1
                C1, C2, C3 = 2 * r1, 2 * r2, 2 * r3

                D_alpha = abs(C1 * alpha[j] - wolves[i, j])
                D_beta = abs(C2 * beta[j] - wolves[i, j])
                D_delta = abs(C3 * delta[j] - wolves[i, j])

                X1 = alpha[j] - A1 * D_alpha
                X2 = beta[j] - A2 * D_beta
                X3 = delta[j] - A3 * D_delta

                wolves[i, j] = (X1 + X2 + X3) / 3
                wolves[i, j] = 1 if wolves[i, j] > 0.5 else 0  # Convert to binary

        # Debug: Print fitness and selected features
        print(f"Iteration {iteration + 1}: Best Fitness = {fitness[sorted_indices[0]]}")
        print(f"Selected Features: {np.where(alpha > 0.5)[0]}")

    best_wolf = alpha
    selected_features = [i for i in range(len(best_wolf)) if best_wolf[i] > 0.5]
    return selected_features

# Apply BGWO for feature selection
selected_features = bgwo_optimize(train_latent, y_train_np)

# Train SVM on selected features
svm_model = SVC(random_state=42)
svm_model.fit(train_latent[:, selected_features], y_train_np)
svm_preds = svm_model.predict(test_latent[:, selected_features])

# Evaluate SVM
svm_acc = accuracy_score(y_test_np, svm_preds)
svm_conf_matrix = confusion_matrix(y_test_np, svm_preds)
svm_report = classification_report(y_test_np, svm_preds)

print('SVM Performance with BGWO-selected features:')
#print('Confusion Matrix\n', svm_conf_matrix)
print('Report\n', svm_report)
print(f"SVM Accuracy on BGWO-selected features: {svm_acc * 100:.4f}%")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.2215
Epoch [2/50], Loss: 0.8043
Epoch [3/50], Loss: 0.6230
Epoch [4/50], Loss: 0.5510
Epoch [5/50], Loss: 0.4811
Epoch [6/50], Loss: 0.4686
Epoch [7/50], Loss: 0.4504
Epoch [8/50], Loss: 0.4388
Epoch [9/50], Loss: 0.4179
Epoch [10/50], Loss: 0.4466
Epoch [11/50], Loss: 0.3906
Epoch [12/50], Loss: 0.3832
Epoch [13/50], Loss: 0.3908
Epoch [14/50], Loss: 0.4200
Epoch [15/50], Loss: 0.3742
Epoch [16/50], Loss: 0.4072
Epoch [17/50], Loss: 0.4047
Epoch [18/50], Loss: 0.4285
Epoch [19/50], Loss: 0.3995
Epoch [20/50], Loss: 0.4003
Epoch [21/50], Loss: 0.4382
Epoch [22/50], Loss: 0.3827
Epoch [23/50], Loss: 0.3936
Epoch [24/50], Loss: 0.3395
Epoch [25/50], Loss: 0.3741
Epoch [26/50], Loss: 0.3879
Epoch [27/50], Loss: 0.4029
Epoch [28/50], Loss: 0.3791
Epoch [29/50], Loss: 0.3944
Epoch [30/50], Loss: 0.3959
Epoch [31/50], Loss: 0.4152
Epoch [32/50], Loss: 0.4150
Epoch [33/50], Loss: 0.3931
Epoch [34/50], Loss: 0.3805
Epoch [35/50], Loss: 0.3424


ValueError: Found array with 0 feature(s) (shape=(354, 0)) while a minimum of 1 is required by SVC.